# ResNet50: experimento GVLiD limpio

**Primero:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 o superior).
Subir `gvlid_clean_colab.zip` a Mi unidad en Google Drive. No requiere los originales ni los datasets anteriores.

Configuración fija: ImageNet1K V2, 224×224, Adam, batch 32, seed 42, warmup 8 épocas a 0.001 y fine-tuning 15 a 0.0001. Selección por Macro F1 de validación **controlada**. No se utiliza validation natural para seleccionar checkpoints.

El baseline original tiene checkpoint e historial, pero carece de evidencia de procedencia vinculada a sus splits. Este notebook permite reentrenar uno comparable sobre las 3.250 imágenes controladas preservadas. No utiliza las métricas anteriores.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ZIP_PATH = Path('/content/drive/MyDrive/gvlid_clean_colab.zip')
WORK = Path('/content/gvlid_clean_experiment')
# Elegir una carpeta NUEVA por experimento. Modelos y logs se guardan en Drive.
RUN = Path('/content/drive/MyDrive/resultados_comparacion_clean')
RETRAIN_BASELINE = True  # Necesario para una comparación completa sin baseline original verificado.


In [ ]:
import hashlib, json, zipfile
if WORK.exists():
    raise FileExistsError('WORK ya existe; usar un entorno nuevo o elegir otra carpeta.')
WORK.mkdir()
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in z.namelist():
        target = (WORK / name).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('Ruta no válida en ZIP')
    z.extractall(WORK)
bundle = json.loads((WORK / 'colab_bundle_manifest.json').read_text())
for relative, expected in bundle['files_sha256'].items():
    assert hashlib.sha256((WORK / relative).read_bytes()).hexdigest() == expected, relative
print('Paquete verificado:', len(bundle['files_sha256']), 'archivos')


In [ ]:
%pip install -q ImageHash==4.3.2 Pillow scikit-learn matplotlib
import os, sys, subprocess, torch
os.chdir(WORK)
assert torch.cuda.is_available(), 'Activar GPU en Colab antes de continuar.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name())
RUNNER = str(WORK / 'scripts/run_gvlid_clean_experiment.py')
def run_action(action, *extra):
    subprocess.run([sys.executable, '-u', RUNNER, action, '--run-dir', str(RUN), *extra], check=True)


## Validación obligatoria
Recalcula SHA-256 y pHash de las copias actuales. Comprueba el inventario completo, 1.434/307/307 naturales, 3.250/406/406 controladas, cuatro clases, grupos confirmados y ausencia de conflictos incluidos. Si falla, detenerse; no corregir el dataset silenciosamente.


In [ ]:
run_action('preflight')


## Entrenamiento híbrido
Entrena **4.684 imágenes** (3.250 PlantVillage + 1.434 GVLiD). La GPU descarga los pesos oficiales de torchvision. Se ejecuta el `train.py` existente sin cambiar arquitectura ni hiperparámetros.

No ejecutar otra vez sobre la misma carpeta. Una desconexión puede interrumpir el trabajo: se conserva el mejor checkpoint, pero el pipeline original no guarda estado de optimizador para reanudar exactamente. Si falla, usar otra carpeta RUN y repetir las 23 épocas completas, sin evaluar el checkpoint parcial.


In [ ]:
run_action('train')


## Baseline comparable
La siguiente celda prepara una copia exclusiva de las imágenes `controlled_*` del dataset limpio y entrena con la misma configuración. No usa imágenes naturales. Si se omite, se evalúa solo el híbrido y queda explícitamente pendiente la comparación.


In [ ]:
if RETRAIN_BASELINE:
    run_action('prepare-baseline')
    run_action('train', '--baseline')


## Evaluación final — ejecutar una vez
Usa el mejor checkpoint por validación controlada. Evalúa 406 imágenes controladas y exactamente las mismas 307 naturales para ambos modelos. Genera CSV, JSON con cada predicción y matrices PNG absolutas y normalizadas. **No ajustar el entrenamiento después de mirar el test.** `vl_black_rot` tiene solo 14 imágenes.


In [ ]:
run_action('evaluate')


In [ ]:
import pandas as pd
from IPython.display import display, Image
EVAL = RUN / 'evaluation'
for filename in ['comparacion_global_natural.csv', 'comparacion_por_clase_natural.csv',
                 'metricas_hibrido_controlado.csv', 'metricas_hibrido_natural.csv', 'metricas_baseline_natural.csv']:
    path = EVAL / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path))
for model in ['hibrido_natural', 'baseline_natural']:
    path = EVAL / f'matriz_confusion_{model}.png'
    if path.exists():
        display(Image(filename=str(path)))
print('Resultados persistentes:', RUN)


In [ ]:
# Archivo descargable de métricas, logs y manifiestos; checkpoints permanecen en Drive.
import zipfile
archive = RUN / 'resultados_metricas.zip'
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file() and path.suffix in {'.json', '.csv', '.png', '.log', '.txt'}:
            z.write(path, str(path.relative_to(RUN)))
print(archive)
